In [8]:
import re
from sys import api_version  #load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
#Create an API client
from anthropic import Anthropic

client = Anthropic()

model = "claude-haiku-4-5-20251001"

In [10]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    asssistant_message = {"role": "assistant", "content": text}
    messages.append(asssistant_message)


def chat(messages, system=None, termperature=0.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": termperature,
        "stop_sequences": stop_sequences,
    }
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text



In [48]:
import json

def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON or Regex specifically for AWS-related tasks. Generate an array of JSON where each represent a task that requires python,json or regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
            "format": "json" or "regex" or "python"
            "solution_criteria": "Key criteria that falls in line with AWS best practices"
        },
        ...additional tasks
    ]
    ```
    *Focus on tasks that can be solved by writing a single Python function, a single json object, a single Regex.
    *Focus on tasks that do not require writing much code
    *no details or examples, just tasks

    Please generate 5 objects
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages,stop_sequences=["```"], termperature=1.0)
    return json.loads(text)

In [49]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [46]:
def run_prompt(test_case):
    prompt = f"""
You are a senior AWS developer. You are tasked with writing code to solve a specific task.
Please solve the following task:

{test_case["task"]}

*Respond only with Python, JSON or plain Regex
*Do not add any comments or explanations
*Keep your code short and concise
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, f"```{test_case['format']}")
    text = chat(messages, stop_sequences=["```"])
    return text

In [50]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution:

Original Task:
<task>
{test_case["task"]}
</task>

Solution to evaluate:
<solution>
{output}
</solution>

Evaluate the solution based on the following criteria:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specified order:
-"strengths": An array or 1-3 key strengths
-"weaknesses": An array of 1-3 key areas that need improvement
-"reasonong": A concise explanation of your overall assessment
-"score": A number between 1 and 10 representing your overall evaluation

Respond with JSON only. Keep your response short and concise.
Example output:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [30]:
import ast
import re

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "regex":
        return validate_regex(response)
    elif format == "python":
        return validate_python(response)


In [29]:
def run_test_case(test_case):
    output = run_prompt(test_case)

    #
    model_grade = grade_by_model(test_case, output)
    sytnax_grade = grade_syntax(output, test_case)
    score = (model_grade["score"] + sytnax_grade)/2
    reasoning = model_grade["reasoning"]

    return {
        "output": output,
        "test_case":test_case,
        "score":score,
        "reasoning":reasoning,
    }

In [51]:
def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [52]:
from statistics import mean

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
avg_score = mean([result["score"] for result in results])

print(json.dumps(results, indent=2))
print("="*20)
print("Average Score:", avg_score)


[
  {
    "output": "\n{\n  \"Bucket\": \"my-bucket\",\n  \"VersioningConfiguration\": {\n    \"Status\": \"Enabled\"\n  },\n  \"PublicAccessBlockConfiguration\": {\n    \"BlockPublicAcls\": true,\n    \"IgnorePublicAcls\": true,\n    \"BlockPublicPolicy\": true,\n    \"RestrictPublicBuckets\": true\n  }\n}\n",
    "test_case": {
      "task": "Create a configuration object for an S3 bucket with versioning enabled and public access blocked",
      "format": "json",
      "solution_criteria": "Must include bucket versioning configuration and public access block settings that comply with AWS security best practices"
    },
    "score": 8.5,
    "reasoning": "The solution correctly implements the core requirements with proper versioning and comprehensive public access blocking. However, it omits additional security best practices such as encryption and logging that would strengthen the overall security posture of the S3 bucket configuration."
  },
  {
    "output": "\n^arn:aws:iam::[0-9]{